# Grounded Answering Demo

Thin educational client over the production interfaces added by the
context-grounded-answering milestone. Every call below goes through the same
public surfaces `engrag-ask` uses (`pipelines.answering_config`,
`pipelines.answering_pipeline`, `pipelines.retrieval_config`) — nothing here
reimplements retrieval, context building, prompting, generation, or grounding
validation.

**What this demonstrates:**
1. Loading and validating the answering + retrieval configuration.
2. Running retrieval and inspecting the ranked chunks it returns.
3. Building a token-budgeted, citable context from those chunks.
4. Reading the citation mapping (citation ID -> chunk/page/section).
5. `engrag-ask validate` — checking Ollama/retrieval/config without generating.
6. Asking one real, answerable question and inspecting the grounded answer.
7. Asking one real, out-of-domain question and confirming a correct refusal.
8. The grounding report, token usage, and latency for a real answer.

**What this does *not* do:** it never prints the system prompt or any hidden
reasoning (`think: false` is enforced by configuration) — only the final
structured answer and its grounding report.


In [1]:
import json
import os

from engineering_rag.utils.paths import repo_root

ROOT = repo_root()
os.chdir(ROOT)  # so relative paths in YAML profiles resolve against the repo root, matching the CLI
print("Working directory:", ROOT)

Working directory: E:\engineering-rag-parser


In [2]:
from engineering_rag.pipelines.answering_config import load_answering_config
from engineering_rag.pipelines.answering_pipeline import (
    run_ask_pipeline,
    run_context_pipeline,
    validate_all,
)
from engineering_rag.pipelines.retrieval_config import load_retrieval_config

ANSWERING_PROFILE = "configs/answering_production.yaml"
RETRIEVAL_PROFILE = "configs/retrieval_production.yaml"

answering_config = load_answering_config(ANSWERING_PROFILE)
retrieval_config = load_retrieval_config(RETRIEVAL_PROFILE)

print("answering model:", answering_config.ollama.model)
print("think (must be False):", answering_config.ollama.think)
print("context_window_tokens:", answering_config.ollama.context_window_tokens)
print("max_context_tokens:", answering_config.context_builder.max_context_tokens)
print("prompt_version:", answering_config.answering.prompt_version)
print("config_hash:", answering_config.config_hash()[:16])

answering model: qwen3:8b
think (must be False): False
context_window_tokens: 8192
max_context_tokens: 5000
prompt_version: 1.0.0
config_hash: 26438b14d3ebf607


## 1. Validate the environment

Checks Ollama reachability/version/model/digest, the retrieval database, and
the token-budget/prompt-contract configuration. **Never generates an
answer.**


In [3]:
report = validate_all(answering_config, retrieval_config)
print(json.dumps(report.as_dict(), indent=2)[:2000])
print()
print(
    "Overall PASS"
    if report.passed
    else "Overall FAIL (see checks above -- expected until Ollama is installed/running)"
)

Ollama GET /api/version connection error (attempt 1/2): [WinError 10061] No connection could be made because the target machine actively refused it


{
  "status": "FAIL",
  "ollama": {
    "status": "FAIL",
    "checks": [
      {
        "check_id": "ollama_reachable",
        "passed": false,
        "summary": "http://127.0.0.1:11434: unreachable"
      }
    ]
  },
  "retrieval": {
    "status": "PASS",
    "checks": [
      {
        "check_id": "chroma_path_exists",
        "passed": true,
        "summary": "data\\output\\databases\\chroma exists"
      },
      {
        "check_id": "collection_exists",
        "passed": true,
        "summary": "collection 'engineering_documents_v1' found"
      },
      {
        "check_id": "collection_not_empty",
        "passed": true,
        "summary": "collection count = 122"
      },
      {
        "check_id": "embedding_dimension_matches_profile",
        "passed": true,
        "summary": "stored=768, profile expects=768"
      },
      {
        "check_id": "distance_metric_is_cosine",
        "passed": true,
        "summary": "stored='cosine'"
      },
      {
        "check_

## 2. Run retrieval and build context

`run_context_pipeline` calls the *existing* retrieval pipeline
(`run_hybrid_search`), then builds a `ContextPackage`: deduplicated,
token-budgeted, with citation IDs assigned only after final selection.


In [4]:
QUESTION = "What activities are performed during the FEED phase?"

retrieval_response, context = run_context_pipeline(
    QUESTION,
    answering_config,
    retrieval_config,
    retrieval_mode="vector",
)

print(
    f"Retrieved {retrieval_response.returned_count} candidate chunk(s) via mode={retrieval_response.retrieval_mode}"
)
for hit in retrieval_response.hits[:5]:
    print(f"  rank={hit.rank} chunk_id={hit.chunk_id} pages={hit.page_numbers} section={hit.section_title!r}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Retrieved 5 candidate chunk(s) via mode=vector
  rank=1 chunk_id=chunk_43856312430296c3 pages=[1] section='Project Lifecycle Phasing and C&I Scope Evolution'
  rank=2 chunk_id=chunk_359f3f41bfa09184 pages=[1] section='Phase 2: Front-End Engineering Design (FEED) and Feasibility'
  rank=3 chunk_id=chunk_caa8f92d1e40f22f pages=[7] section='2.2 Phase 2: Front-End Engineering Design (FEED) and Feasibility'
  rank=4 chunk_id=chunk_ab16bedfbf6f078b pages=[1] section='Phase 3: Detailed Engineering Design (Execution Core)'
  rank=5 chunk_id=chunk_df5ca6e2fa22b880 pages=[6] section='Activity: Defining Operational Needs and User Functional Requirements (UFR)'


In [5]:
print(f"Selected {context.total_sources_selected} of {context.total_candidates_received} candidates")
print(f"Context tokens: {context.context_token_count} / budget {context.token_budget}")
print(f"Excluded: {len(context.excluded_candidates)}")
for excluded in context.excluded_candidates:
    print(f"  excluded {excluded.chunk_id}: {excluded.reason} -- {excluded.detail}")

Selected 10 of 5 candidates
Context tokens: 1268 / budget 5000
Excluded: 1
  excluded chunk_fc42b5d2866dfc5f: max_sources_reached -- neighbor expansion: max_sources already reached


## 3. Citation mapping

Every selected source's answer-local citation ID (`S1`, `S2`, ...) maps back
to its full provenance -- this is what lets the final answer say
`"...control philosophy [S1]."` and have that be independently checkable.


In [6]:
for source in context.selected_sources:
    print(
        f"[{source.citation_id}] chunk={source.chunk_id} file={source.source_filename} "
        f"pages={source.page_numbers} section={source.section_title!r} "
        f"neighbor={source.is_neighbor} tokens={source.token_count}"
    )

[S1] chunk=chunk_43856312430296c3 file=Instrumentation-and-Control-Engineering.pdf pages=[1] section='Project Lifecycle Phasing and C&I Scope Evolution' neighbor=False tokens=78
[S2] chunk=chunk_359f3f41bfa09184 file=Instrumentation-and-Control-Engineering.pdf pages=[1] section='Phase 2: Front-End Engineering Design (FEED) and Feasibility' neighbor=False tokens=48
[S3] chunk=chunk_caa8f92d1e40f22f file=Instrumentation-and-Control-Engineering.pdf pages=[7] section='2.2 Phase 2: Front-End Engineering Design (FEED) and Feasibility' neighbor=False tokens=50
[S4] chunk=chunk_ab16bedfbf6f078b file=Instrumentation-and-Control-Engineering.pdf pages=[1] section='Phase 3: Detailed Engineering Design (Execution Core)' neighbor=False tokens=52
[S5] chunk=chunk_df5ca6e2fa22b880 file=Instrumentation-and-Control-Engineering.pdf pages=[6] section='Activity: Defining Operational Needs and User Functional Requirements (UFR)' neighbor=False tokens=70
[S6] chunk=chunk_12853b0e951df7d3 file=Instrumentation

## 4. Ask a real, answerable question

Requires a local Ollama server with `qwen3:8b` installed and reachable at
`http://127.0.0.1:11434` (see `docs/answering/OLLAMA_SETUP.md`). If Ollama is
not running, this cell raises `OllamaConnectionError` -- that is the correct,
honest behavior; this notebook does not fabricate a successful answer when
Ollama is unavailable.


In [7]:
_rr, _ctx, answer, _trace, run_dir = run_ask_pipeline(
    QUESTION,
    answering_config,
    retrieval_config,
    retrieval_mode="vector",
)

print("status:", answer.status)
print()
print(answer.answer)
print()
print("citations:")
for c in answer.citations:
    print(f"  [{c.citation_id}] {c.source_filename} p.{c.page_numbers} {c.section_title or ''}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Ollama GET /api/tags connection error (attempt 1/2): [WinError 10061] No connection could be made because the target machine actively refused it


Could not resolve installed model digest before generation: Could not connect to Ollama at http://127.0.0.1:11434 after 2 attempt(s): [WinError 10061] No connection could be made because the target machine actively refused it


Ollama POST /api/chat connection error (attempt 1/2): [WinError 10061] No connection could be made because the target machine actively refused it


status: generation_failed



citations:


## 5. Grounding report, token use, and latency


In [8]:
print("grounding status:", answer.validation.status)
print("checks_passed:", answer.validation.checks_passed)
print("checks_failed:", answer.validation.checks_failed)
print("warnings:", answer.validation.warnings)
print()
print("model:", answer.model_tag, "digest:", answer.model_digest)
print("prompt_token_count:", answer.prompt_token_count)
print("answer_token_count:", answer.answer_token_count)
print("total_latency_s:", answer.total_latency_s)
print("stage_latencies_s:", answer.stage_latencies_s)
print()
print("artifact run directory:", run_dir.root if run_dir else None)

grounding status: FAIL
checks_passed: []
checks_failed: ['llm_connection_or_timeout_failure']
warnings: []

model: qwen3:8b digest: None
prompt_token_count: None
answer_token_count: None
total_latency_s: 4.1198
stage_latencies_s: {'generation': 4.119475}

artifact run directory: data\output\answering\4baa4421ebd44397a062b0331c382f09


## 6. Ask a real, out-of-domain question -- confirm correct refusal

The indexed corpus is an engineering-process document with no sports or
financial data. The system must refuse rather than answer from the model's
own outside knowledge.


In [9]:
_rr2, _ctx2, refusal_answer, _trace2, _run_dir2 = run_ask_pipeline(
    "Who won the FIFA World Cup in 2030?",
    answering_config,
    retrieval_config,
    retrieval_mode="vector",
)

print("status:", refusal_answer.status)
print(refusal_answer.answer)
assert refusal_answer.status == "insufficient_evidence", "expected a refusal for this out-of-domain question"
print()
print("Correctly refused: no citations were fabricated, no outside knowledge was used.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Ollama GET /api/tags connection error (attempt 1/2): [WinError 10061] No connection could be made because the target machine actively refused it


Could not resolve installed model digest before generation: Could not connect to Ollama at http://127.0.0.1:11434 after 2 attempt(s): [WinError 10061] No connection could be made because the target machine actively refused it


Ollama POST /api/chat connection error (attempt 1/2): [WinError 10061] No connection could be made because the target machine actively refused it


status: generation_failed



AssertionError: expected a refusal for this out-of-domain question

## Summary

- Configuration, retrieval, and context building are exercised through the
  same public interfaces `engrag-ask context`/`validate` use.
- Citation IDs are assigned only after final selection and map back to full
  provenance (file, pages, section, neighbor status).
- A real answerable question produces a cited, grounding-validated answer;
  a real out-of-domain question produces an explicit refusal -- never a
  fabricated answer from the model's outside knowledge.
- No hidden reasoning is ever printed or stored: `think: false` is enforced
  by `OllamaConfig`'s validator, and `AnswerTrace`/every artifact file
  capture only the final structured JSON.

See `docs/answering/` for the full architecture, security/grounding model,
evaluation methodology, and Ollama setup instructions.
